<h1 style="text-align: center;">hybraut watchdog</h1>

The runtime system is monitored and managed by a **hybraut-executor-watchdog (FSM)**. This FSM listens to runtime events and responds accordingly to maintain mission continuity and enable it to automate part of the **hybraut_ros2** lifecycle, enable it to recover from failures, and safely shut down in critical conditions. It helps enforce lifecycle safety and determinism across automaton operations.

When specific events are published (e.g., mode transitions, errors, mission completion), the FSM interprets these and updates the automaton's state. Each FSM state is also associated with a corresponding **status message**, which is published to inform external systems about the automaton's condition.

![hybraut_watchdog_fsm_png](../.github/diagrams/hybraut_watchdog_fsm_flowchart.png)

## Setup

The code below is a simple setup of a mock node containing the **hybraut-exeuctor-watchdog (FSM)** I/O ros2 publishers and subscriptions that are required for the automaton to transition and output statuses associated with **FSM** states

In [29]:
import rclpy
from rclpy.node import Node
from rclpy.qos import QoSProfile, qos_profile_system_default
import os
import threading
import time
from rclpy.executors import MultiThreadedExecutor, Executor
from rclpy.callback_groups import ReentrantCallbackGroup
from hybraut_executor_watchdog import FSM
from hybraut_interfaces.msg import AutomatonEvents, AutomatonStatus
from hybraut_executor_watchdog.hybraut_consts import EventEnum, StatusEnum

if rclpy.ok():
    rclpy.shutdown()


rclpy.init()


QOS = QoSProfile(depth=10, reliability=qos_profile_system_default.reliability)

try:
    executor: Executor = MultiThreadedExecutor(num_threads=os.cpu_count())
    node = Node("hybraut_watchdog_demo")
    executor.add_node(node)
    
    # Start executor in background thread
    thread = threading.Thread(target=executor.spin, daemon=True)
    thread.start()
    
    # Create FSM instance
    fsm = FSM(node=node, cb_group=ReentrantCallbackGroup(), qos=QOS)
    
    # Create event publisher for testing
    event_publisher = node.create_publisher(
        AutomatonEvents, "/automaton/events", 
        qos_profile=QOS, callback_group=ReentrantCallbackGroup()
    )

    current_state = [fsm.get_current_state().value]

    def status_callback(msg: AutomatonStatus):
        current_state.append(msg.type)
        
    status_subscription = node.create_subscription(
        AutomatonStatus, "/automaton/status",
        status_callback, qos_profile=QOS, callback_group=ReentrantCallbackGroup()
    )
    
    # Wait for connections
    time.sleep(1.0)
except Exception as e:
    rclpy.shutdown()
    exit(1)

below is a function for running through test event list and verifying the tests executed correctly

In [30]:
from typing import List, Tuple

def execute_test_events(fsm: FSM, test_events: List[Tuple[EventEnum, str]], expected_status: List[Tuple[StatusEnum, str]]) -> None:
    """Execute a series of test events on the FSM and print the current state and valid transitions after each event."""
    
    for (event_type, message), (status_type, status_message) in zip(test_events, expected_status):
        event_msg = AutomatonEvents(type=event_type.value, message=message)
        event_publisher.publish(event_msg)
        time.sleep(0.01)

## Basic Happy Path Test

This test tests a valid test. 
Active -> TRANSITIONING -> ACTIVE -> MISSION_COMPLETE

In [ ]:
test_events = [
    (EventEnum.TRANSITION_GUARD_ENABLED, "mode guard activated"),
    (EventEnum.TRANSITION_COMPLETE, "transition completed"),
    (EventEnum.TRANSITION_GUARD_ENABLED, "mode guard activated"),
    (EventEnum.TRANSITION_COMPLETE, "transition completed"),
    (EventEnum.TRANSITION_GUARD_ENABLED, "mode guard activated"),
    (EventEnum.TRANSITION_COMPLETE, "transition completed"),
    (EventEnum.TRANSITION_GUARD_ENABLED, "mode guard activated"),
    (EventEnum.TRANSITION_COMPLETE, "transition completed"),
    (EventEnum.TRANSITION_GUARD_ENABLED, "mode guard activated"),
    (EventEnum.TRANSITION_COMPLETE, "transition completed"),
    (EventEnum.MISSION_COMPLETE, "mission has completed"),
]

expected_status = [
    (StatusEnum.TRANSITIONING, ""),
    (StatusEnum.ACTIVE, ""),
    (StatusEnum.TRANSITIONING, ""),
    (StatusEnum.ACTIVE, ""),
    (StatusEnum.TRANSITIONING, ""),
    (StatusEnum.ACTIVE, ""),
    (StatusEnum.TRANSITIONING, ""),
    (StatusEnum.ACTIVE, ""),
    (StatusEnum.TRANSITIONING, ""),
    (StatusEnum.TRANSITIONING, ""),
    (StatusEnum.MISSION_COMPLETE, "")
]

execute_test_events(fsm, test_events, expected_status)

time.sleep(1.0)
print (current_state)

[INFO] [1754581388.846113375] [hybraut_watchdog_demo]: Processing event: EventEnum.TRANSITION_GUARD_ENABLED
[INFO] [1754581388.847681265] [hybraut_watchdog_demo]: Guard enabled, transitioning: EventEnum.TRANSITION_GUARD_ENABLED
[INFO] [1754581388.850510055] [hybraut_watchdog_demo]: Status received: hybraut_interfaces.msg.AutomatonStatus(type=3, message='State changed to StatusEnum.TRANSITIONING', stamp=builtin_interfaces.msg.Time(sec=1754581388, nanosec=847775585))
[INFO] [1754581388.851496275] [hybraut_watchdog_demo]: Published status: StatusEnum.TRANSITIONING
[INFO] [1754581388.854164955] [hybraut_watchdog_demo]: Fired trigger: enable_guard
[INFO] [1754581388.855095015] [hybraut_watchdog_demo]: Processing event: EventEnum.TRANSITION_COMPLETE
[INFO] [1754581388.855643115] [hybraut_watchdog_demo]: Guard enabled, transitioning: EventEnum.TRANSITION_COMPLETE
[INFO] [1754581388.858955115] [hybraut_watchdog_demo]: Status received: hybraut_interfaces.msg.AutomatonStatus(type=2, message='Sta

[2, 3, 2, 3, 2]


[INFO] [1754581389.134849575] [hybraut_watchdog_demo]: Fired trigger: enable_guard
[INFO] [1754581389.139562785] [hybraut_watchdog_demo]: Status received: hybraut_interfaces.msg.AutomatonStatus(type=2, message='State changed to StatusEnum.ACTIVE', stamp=builtin_interfaces.msg.Time(sec=1754581388, nanosec=900286085))
[INFO] [1754581389.143212465] [hybraut_watchdog_demo]: Processing event: EventEnum.TRANSITION_COMPLETE
[INFO] [1754581389.203354785] [hybraut_watchdog_demo]: Published status: StatusEnum.ACTIVE
[INFO] [1754581389.206367895] [hybraut_watchdog_demo]: Fired trigger: complete_transition
[INFO] [1754581389.206627635] [hybraut_watchdog_demo]: Processing event: EventEnum.TRANSITION_GUARD_ENABLED
[INFO] [1754581389.208427725] [hybraut_watchdog_demo]: Processing event: EventEnum.TRANSITION_COMPLETE
[INFO] [1754581389.217665685] [hybraut_watchdog_demo]: Processing event: EventEnum.TRANSITION_GUARD_ENABLED
[INFO] [1754581389.220398895] [hybraut_watchdog_demo]: Fired trigger: complete_

[INFO] [1754581389.379144135] [hybraut_watchdog_demo]: Published status: StatusEnum.ACTIVE
[INFO] [1754581389.389509525] [hybraut_watchdog_demo]: Status received: hybraut_interfaces.msg.AutomatonStatus(type=2, message='State changed to StatusEnum.ACTIVE', stamp=builtin_interfaces.msg.Time(sec=1754581389, nanosec=223914115))
[INFO] [1754581389.400723345] [hybraut_watchdog_demo]: Status received: hybraut_interfaces.msg.AutomatonStatus(type=2, message='State changed to StatusEnum.ACTIVE', stamp=builtin_interfaces.msg.Time(sec=1754581389, nanosec=236970235))
[INFO] [1754581389.404905735] [hybraut_watchdog_demo]: Published status: StatusEnum.ACTIVE
[INFO] [1754581389.405978985] [hybraut_watchdog_demo]: Status received: hybraut_interfaces.msg.AutomatonStatus(type=2, message='State changed to StatusEnum.ACTIVE', stamp=builtin_interfaces.msg.Time(sec=1754581389, nanosec=276962365))
[INFO] [1754581389.406964215] [hybraut_watchdog_demo]: Published status: StatusEnum.ACTIVE
[INFO] [1754581389.411

## Error Handling Tests: 

### Recoverable Error Path
 ON_VALID_MISSION_REQUEST -> ACTIVE -> RECOVERABLE_ERROR -> ERROR -> RECOVERING -> RECOVERED -> ACTIVE -> MISSION_COMPLETE -> MISSION_COMPLETE

### Critical Failure Path
ON_VALID_MISSION_REQUEST → ACTIVE
RECOVERABLE_ERROR → ERROR
CRITICAL_FAILURE → FATAL
SHUTDOWN → (terminal state)

Recovery Failure Path:

ON_VALID_MISSION_REQUEST → ACTIVE
RECOVERABLE_ERROR → ERROR
ATTEMPT_FIX → RECOVERING
RECOVERY_FAILED → FATAL
SHUTDOWN → (terminal state)

#### Error From Transitioning
ON_VALID_MISSION_REQUEST → ACTIVE
TRANSITION_GUARD_ENABLED → TRANSITIONING
RECOVERABLE_ERROR → ERROR
ATTEMPT_FIX → RECOVERING
RECOVERED → ACTIVE

In [6]:
if rclpy.ok():
    rclpy.shutdown()